<a href="https://colab.research.google.com/github/esmajasa/MLFlyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/esmajasa/MLFlyrank/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

In [2]:
import os
import getpass
import duckdb
import numpy as np
import pandas as pd

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except (ImportError, KeyError, RuntimeError):
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)
del safe_token

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet'"
    f")"
)

print("Connected to the March 2026 warehouse partition.")

Connected to the March 2026 warehouse partition.


1. unit of analysis: onw row represents one report date, one client, and one content item. After aggregation, one feature-frame row represents one client-content pair
2. I'll use the March partition of fact_content_daily_performance
3. The feature window is March 1-21, 2026. The decision moment is the beginning of March 22. The outcome window is March 22-31.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Purpose |
|---|---|---|
| Features | `feature_ctr`, `log_feature_impressions`, `feature_avg_position`, `feature_position_volatility`, `feature_active_days` | Exactly five measurements known before March 22 |
| Label/proxy | `future_low_ctr_opportunity` | Whether later CTR is low relative to pages in a similar later-position band |
| Context | `client_hash_id`, `content_hash_id`, `target_position_band` | Used for identification, grouping and interpretation—not as model inputs |
| Excluded | `target_clicks`, `target_ctr` | They come from March 22–31 and directly construct the proxy |
| Availability | `ga4_data_available` | Used as a filter, not as a model feature |

Exclusing target_ctr from the honest fiture list since it is measured after the decision moment and directly determines the low-CTR proxy.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verificatino Query 1:
The documented source grain is one row per report date, client and content item. If the query returns zero rows, no duplicate source keys have been found

In [3]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_per_key
    FROM {MARCH_DAILY}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

display(grain_check)

if grain_check.empty:
    print(
        "Grain holds: no duplicate "
        "date-client-content keys were found."
    )
else:
    print("Warning: duplicate source keys were found.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,rows_per_key


Grain holds: no duplicate date-client-content keys were found.


Verification Query 2: Prove the size of the selected March partition and its actual date span

In [4]:
slice_check = con.sql(f"""
    SELECT
        COUNT(*) AS daily_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {MARCH_DAILY}
""").df()

display(slice_check)

,daily_rows,clients,content_items,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


Verification quary 3: GA4 values before a client's GA4 tracking start can be zero-filled rather than measured. So check availability using ga4_data_available IS TRUE. The final column requires positive GSC impressions for the CTR slice.

In [5]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS all_daily_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
              AND gsc_impressions > 0
        ) AS eligible_daily_rows

    FROM {MARCH_DAILY}
""").df()

availability_check["eligible_share_pct"] = (
    100
    * availability_check["eligible_daily_rows"]
    / availability_check["all_daily_rows"]
).round(2)

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_daily_rows,ga4_available_rows,eligible_daily_rows,eligible_share_pct
0,9841378,413966,364347,3.7


Feature construction

In [6]:
page_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature-window measurements: March 1–21
        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-21'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS feature_impressions,

        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-21'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS feature_clicks,

        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-21'
                 AND gsc_avg_position IS NOT NULL
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN report_date <= DATE '2026-03-21'
                     AND gsc_avg_position IS NOT NULL
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS feature_avg_position,

        STDDEV_SAMP(
            CASE
                WHEN report_date <= DATE '2026-03-21'
                 AND gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS feature_position_volatility,

        COUNT(*) FILTER (
            WHERE report_date <= DATE '2026-03-21'
              AND gsc_impressions > 0
        ) AS feature_active_days,

        -- Outcome-window measurements: March 22–31
        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-22'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS target_impressions,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-22'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS target_clicks,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-22'
                 AND gsc_avg_position IS NOT NULL
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN report_date >= DATE '2026-03-22'
                     AND gsc_avg_position IS NOT NULL
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS target_avg_position

    FROM {MARCH_DAILY}

    WHERE ga4_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING
        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-21'
                THEN gsc_impressions
                ELSE 0
            END
        ) >= 100

        AND

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-22'
                THEN gsc_impressions
                ELSE 0
            END
        ) >= 50
""").df()

print("Page-level rows:", len(page_frame))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level rows: 17423


In [8]:
page_frame["feature_ctr"] = (
    100
    * page_frame["feature_clicks"]
    / page_frame["feature_impressions"]
)

page_frame["log_feature_impressions"] = np.log1p(
    page_frame["feature_impressions"]
)

page_frame["target_ctr"] = (
    100
    * page_frame["target_clicks"]
    / page_frame["target_impressions"]
)

FEATURES = [
    "feature_ctr",
    "log_feature_impressions",
    "feature_avg_position",
    "feature_position_volatility",
    "feature_active_days",
]

assert len(FEATURES) == 5

display(
    page_frame[
        [
            "client_hash_id",
            "content_hash_id",
            *FEATURES,
        ]
    ].head()
)

,client_hash_id,content_hash_id,feature_ctr,log_feature_impressions,feature_avg_position,feature_position_volatility,feature_active_days
0,client_9958f0a7ae1df715,content_810cf06597918291,0.609756,5.105945,8.634146,1.879816,11
1,client_9958f0a7ae1df715,content_5a77dbf5671c5a65,1.106345,9.494617,4.701061,0.457812,21
2,client_9958f0a7ae1df715,content_278030b007943b07,2.066116,5.493061,6.045455,1.069471,8
3,client_9958f0a7ae1df715,content_237e63fc00c8c7ad,0.419483,8.364508,6.975530,0.648941,11
4,client_9958f0a7ae1df715,content_347fbafb77d3ae37,1.098901,5.613128,21.684982,5.119335,14


Creating the future proxy

In [9]:
position_edges = [0, 3, 10, 20, 50, np.inf]

position_names = [
    "top_3",
    "page_1",
    "striking",
    "page_3_5",
    "deep",
]

page_frame["target_position_band"] = pd.cut(
    page_frame["target_avg_position"],
    bins=position_edges,
    labels=position_names,
)

In [10]:
band_median_target_ctr = (
    page_frame
    .groupby(
        "target_position_band",
        observed=True,
    )["target_ctr"]
    .transform("median")
)

page_frame["future_low_ctr_opportunity"] = (
    page_frame["target_ctr"]
    < band_median_target_ctr
).astype(int)

print(
    page_frame["future_low_ctr_opportunity"]
    .value_counts(dropna=False)
)

print(
    "Positive share:",
    page_frame["future_low_ctr_opportunity"].mean().round(3),
)

future_low_ctr_opportunity
0    8769
1    8654
Name: count, dtype: int64
Positive share: 0.497


Deliberately leaking the answer

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

model_frame = page_frame.dropna(
    subset=[
        "target_position_band",
        "future_low_ctr_opportunity",
    ]
).copy()

train_index, test_index = train_test_split(
    np.arange(len(model_frame)),
    test_size=0.25,
    random_state=42,
    stratify=model_frame["future_low_ctr_opportunity"],
)

y_train = model_frame.iloc[
    train_index
]["future_low_ctr_opportunity"]

y_test = model_frame.iloc[
    test_index
]["future_low_ctr_opportunity"]

In [12]:
def quick_score(feature_columns):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=150,
            random_state=42,
            n_jobs=-1,
        ),
    )

    model.fit(
        model_frame.iloc[train_index][feature_columns],
        y_train,
    )

    predictions = model.predict(
        model_frame.iloc[test_index][feature_columns]
    )

    return balanced_accuracy_score(
        y_test,
        predictions,
    )

In [13]:
honest_score = quick_score(FEATURES)

print(
    "Honest five-feature balanced accuracy:",
    round(honest_score, 3),
)

Honest five-feature balanced accuracy: 0.727


In [14]:
LEAKY_FEATURES = FEATURES + ["target_ctr"]

leaky_score = quick_score(LEAKY_FEATURES)

print(
    "Leaky balanced accuracy:",
    round(leaky_score, 3),
)

print(
    "Artificial score increase:",
    round(leaky_score - honest_score, 3),
)

Leaky balanced accuracy: 0.953
Artificial score increase: 0.226


In [15]:
honest_frame = model_frame.drop(
    columns=["target_ctr"]
)

assert "target_ctr" not in honest_frame.columns

print(
    "target_ctr removed. "
    "The retained result is the honest score."
)

target_ctr removed. The retained result is the honest score.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This experiment uses only one short feature window and one ten-day outcome window from March 2026. The observed relationship may not generalize across clients, seasons, or later months.
Client history is unbalanced, meaning different clients have different amounts of recorded history. Requiring ga4_data_available IS TRUE also changes which pages and clients survive the filter.
The low-CTR outcome is a defined review proxy. It does not prove that a page is defective, and it cannot show that an editorial change would cause CTR to improve. CTR may also be affected by query mix, competitors, search-result features, brand recognition and user intent.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.